# Multi-format RAG ingestion pipeline
Loads **PDF, DOCX, HTML, CSV and TXT/MD** files from a folder, chunks them, embeds them, and upserts everything into Pinecone.

In [1]:
%pip install -q pypdf docx2txt beautifulsoup4 lxml rank_bm25 \
    langchain langchain-community langchain-text-splitters langchain-huggingface \
    sentence-transformers pinecone groq python-dotenv pandas tqdm

Note: you may need to restart the kernel to use updated packages.


## Imports

In [2]:
import os
import csv
from pathlib import Path
from typing import Callable, Dict, List

from dotenv import load_dotenv
from tqdm.auto import tqdm
import pandas as pd
from langchain_core.documents import Document as LangchainDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    BSHTMLLoader,
)

from pinecone import Pinecone
from groq import Groq

load_dotenv()

C:\Users\Ahmed\Downloads\Med-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Ahmed\AppData\Local\Temp\ipykernel_26256\1828466887.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


True

## Configuration

In [3]:
DATA_DIR = "./data"  # folder containing the raw files to ingest (any mix of types below)

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = "lab-rag-index"
PINECONE_NAMESPACE = "ns1"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
UPSERT_BATCH_SIZE = 100

## Multi-format document loaders
Each loader returns a list of `langchain_core.documents.Document`. Add new extensions to `EXTENSION_LOADERS` to support more formats.

In [4]:
def load_text(path: str) -> List[LangchainDocument]:
    return TextLoader(path, encoding="utf-8").load()


def load_healthcare_csv(path: str) -> List[LangchainDocument]:
    """Loads the healthcare_rag_dataset.csv schema: builds page_content from
    the clinical text fields only (content_text, symptoms, treatments, risk_factors,
    prevention) and keeps the rest of the columns as metadata instead of dumping all
    23 columns into the embedded text (which would bury the signal in noise like
    word_count/reading_level_score and waste context tokens at generation time).
    """
    docs = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            content = "\n".join(
                filter(
                    None,
                    [
                        f"{row.get('title', '')} ({row.get('document_type', '')})",
                        row.get("content_text"),
                        f"Symptoms: {row['symptoms']}" if row.get("symptoms") else None,
                        f"Treatments: {row['treatments']}" if row.get("treatments") else None,
                        f"Risk factors: {row['risk_factors']}" if row.get("risk_factors") else None,
                        f"Prevention: {row['prevention']}" if row.get("prevention") else None,
                    ],
                )
            )
            metadata = {
                "source": path,
                "doc_id": row.get("doc_id"),
                "title": row.get("title"),
                "category": row.get("category"),
                "icd10_code": row.get("icd10_code"),
                "severity_level": row.get("severity_level"),
                "source_name": row.get("source_name"),
                "source_url": row.get("source_url"),
            }
            docs.append(LangchainDocument(page_content=content, metadata=metadata))
    return docs


EXTENSION_LOADERS: Dict[str, Callable[[str], List[LangchainDocument]]] = {
    ".txt": load_text,
    ".md": load_text,
    ".pdf": lambda path: PyPDFLoader(path).load(),
    ".docx": lambda path: Docx2txtLoader(path).load(),
    ".html": lambda path: BSHTMLLoader(path).load(),
    ".htm": lambda path: BSHTMLLoader(path).load(),
    ".csv": load_healthcare_csv,
}


def load_any_file(path: str) -> List[LangchainDocument]:
    ext = Path(path).suffix.lower()
    loader = EXTENSION_LOADERS.get(ext)
    if loader is None:
        print(f"Skipping unsupported file type: {path}")
        return []
    try:
        return loader(path)
    except Exception as exc:
        print(f"Failed to load {path}: {exc}")
        return []


def load_directory(folder: str) -> List[LangchainDocument]:
    all_docs: List[LangchainDocument] = []
    for root, _, files in os.walk(folder):
        for fname in files:
            all_docs.extend(load_any_file(os.path.join(root, fname)))
    return all_docs

## Load every supported file under `DATA_DIR`

In [5]:
raw_documents = load_directory(DATA_DIR)
print(f"Loaded {len(raw_documents)} source document(s) from {DATA_DIR}")

Loaded 409 source document(s) from ./data


## Chunking

In [6]:
MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
    strip_whitespace=True,
    separators=MARKDOWN_SEPARATORS,
)

docs_processed = text_splitter.split_documents(raw_documents)
print(f"Split into {len(docs_processed)} chunk(s)")

Split into 549 chunk(s)


## Embedding model

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4767.51it/s]

## Upsert every chunk into Pinecone (batched)

In [8]:
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

# One stable id per chunk, reused below for BM25 / fusion / reranking / eval.
chunk_ids = [f"vec{i}" for i in range(len(docs_processed))]
id_to_text = {chunk_id: doc.page_content for chunk_id, doc in zip(chunk_ids, docs_processed)}


def batched(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


for batch in tqdm(list(batched(list(zip(chunk_ids, docs_processed)), UPSERT_BATCH_SIZE))):
    vectors = [
        {
            "id": chunk_id,
            "values": embedding_model.embed_query(doc.page_content),
            "metadata": {
                "text": doc.page_content,
                "source": doc.metadata.get("source", "unknown"),
            },
        }
        for chunk_id, doc in batch
    ]
    index.upsert(vectors=vectors, namespace=PINECONE_NAMESPACE)

print("Ingestion complete.")

  0%|          | 0/6 [00:00<?, ?it/s]

 17%|█▋        | 1/6 [00:06<00:31,  6.22s/it]

 33%|███▎      | 2/6 [00:11<00:22,  5.57s/it]

 50%|█████     | 3/6 [00:16<00:16,  5.37s/it]

 67%|██████▋   | 4/6 [00:21<00:10,  5.31s/it]

 83%|████████▎ | 5/6 [00:26<00:05,  5.17s/it]

100%|██████████| 6/6 [00:29<00:00,  4.34s/it]

100%|██████████| 6/6 [00:29<00:00,  4.89s/it]

Ingestion complete.


## Quick retrieval + Groq generation smoke test

In [9]:
groq_client = Groq(api_key=GROQ_API_KEY)

prompt_template = """You are a helpful assistant that answers medical questions using only the provided context.
If the context doesn't contain the answer, say "I don't know".

Context:
{context}

Question: {question}
"""


def ask(question: str, top_k: int = 3) -> str:
    query_vector = embedding_model.embed_query(question)
    results = index.query(
        namespace=PINECONE_NAMESPACE,
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
    )
    context = "\n".join(match["metadata"]["text"] for match in results["matches"])
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt_template.format(context=context, question=question)}],
    )
    return response.choices[0].message.content


print(ask("What are the symptoms of diabetes?"))

The symptoms of Type 2 Diabetes Mellitus include: increased thirst; frequent urination; fatigue; blurred vision; slow-healing sores; unexplained weight loss; numbness or tingling in hands or feet.


## Hybrid search: sparse (BM25) + dense retrieval
Dense retrieval (Pinecone + embeddings) is good at semantic similarity but can miss exact keyword/acronym matches. Sparse retrieval (BM25) is the opposite. Combining both, then reranking the merged candidates, is the standard "hybrid search" recipe.

> Runs in the same session as ingestion above, since it reuses `docs_processed`, `chunk_ids` and `id_to_text` from memory rather than re-fetching everything from Pinecone.

### Sparse retriever (BM25)

In [10]:
import re
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> List[str]:
    return re.findall(r"\w+", text.lower())


bm25_corpus_tokens = [tokenize(doc.page_content) for doc in docs_processed]
bm25 = BM25Okapi(bm25_corpus_tokens)


def sparse_search(query: str, top_k: int = 10) -> List[dict]:
    scores = bm25.get_scores(tokenize(query))
    ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [
        {"id": chunk_ids[i], "score": float(scores[i]), "text": id_to_text[chunk_ids[i]]}
        for i in ranked_idx
    ]

### Dense retriever (Pinecone)

In [11]:
def dense_search(query: str, top_k: int = 10) -> List[dict]:
    query_vector = embedding_model.embed_query(query)
    results = index.query(
        namespace=PINECONE_NAMESPACE,
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
    )
    return [
        {"id": match["id"], "score": float(match["score"]), "text": match["metadata"]["text"]}
        for match in results["matches"]
    ]

### Rank fusion (Reciprocal Rank Fusion)
RRF combines multiple ranked lists without needing their scores to be on the same scale (BM25 scores and cosine similarity aren't comparable directly). Each result's fused score is `sum(1 / (k + rank))` across every list it appears in.

In [12]:
def reciprocal_rank_fusion(rankings: List[List[str]], k: int = 60) -> List[str]:
    fused_scores: Dict[str, float] = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(fused_scores, key=fused_scores.get, reverse=True)


def hybrid_search_fused(query: str, top_k: int = 10, candidate_pool: int = 30) -> List[dict]:
    dense_ids = [r["id"] for r in dense_search(query, candidate_pool)]
    sparse_ids = [r["id"] for r in sparse_search(query, candidate_pool)]
    fused_ids = reciprocal_rank_fusion([dense_ids, sparse_ids])[:top_k]
    return [{"id": doc_id, "text": id_to_text[doc_id]} for doc_id in fused_ids]

### Cross-encoder reranking
RRF fusion is cheap but score-agnostic; a cross-encoder scores each (query, candidate) pair jointly and gives a much better final ordering, at the cost of being slower to run per document — so it's applied only to the small fused candidate set, not the whole corpus.

In [13]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, candidates: List[dict], top_k: int = 5) -> List[dict]:
    pairs = [(query, candidate["text"]) for candidate in candidates]
    scores = reranker.predict(pairs)
    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)
    return sorted(candidates, key=lambda c: c["rerank_score"], reverse=True)[:top_k]


def hybrid_search(query: str, top_k: int = 5, candidate_pool: int = 30) -> List[dict]:
    fused = hybrid_search_fused(query, top_k=candidate_pool, candidate_pool=candidate_pool)
    return rerank(query, fused, top_k=top_k)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2090.59it/s]

## Evaluation: NDCG@k across retrieval strategies
NDCG (Normalized Discounted Cumulative Gain) rewards relevant results appearing near the top of the ranking, with graded relevance support (not just relevant/irrelevant). Define `EVAL_SET` below with real chunk ids from `chunk_ids` and their relevance grade per query (e.g. 2 = highly relevant, 1 = somewhat, 0/absent = not relevant), then compare dense-only, sparse-only, RRF-fused and reranked hybrid search.

In [14]:
import math


def ndcg_at_k(relevances: List[int], k: int) -> float:
    relevances = relevances[:k]
    dcg = sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(relevances))
    ideal = sorted(relevances, reverse=True)
    idcg = sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

In [15]:
# Self-contained imports so this evaluation cell runs even if the top "Imports"
# cell wasn't re-run in the current kernel session.
import pandas as pd
from typing import List

# Real test cases: each maps a natural question to the doc_id(s) in
# healthcare_rag_dataset.csv that should answer it. This dataset has TWO equally
# relevant docs per disease for symptom queries (a "Symptom Guide" and a
# "What are the common symptoms of X?" FAQ) — crediting only one would unfairly
# penalize a retriever for correctly surfacing the other.
TEST_QUERIES = [
    ("What are the symptoms of conjunctivitis (pink eye)?", ["DOC-E6C0C42D", "DOC-EF53E84B"]),
    ("What are common symptoms of allergic rhinitis?", ["DOC-D8F798B2", "DOC-A921FFB8"]),
    ("What does eczema look like and what are its symptoms?", ["DOC-E911986D", "DOC-179D9829"]),
    ("What are the symptoms of hyperlipidemia?", ["DOC-1427E31F", "DOC-CEC0BC16"]),
    ("What are the symptoms of asthma?", ["DOC-2117F5AF", "DOC-5FF9D8C6"]),
    ("What are the symptoms of iron deficiency anemia?", ["DOC-6FF549B5", "DOC-74EC101D"]),
    ("What are the symptoms of type 2 diabetes?", ["DOC-1730475D", "DOC-ECFFDC24"]),
    ("What are the symptoms of a urinary tract infection?", ["DOC-64EEB1E0", "DOC-E2395F2A"]),
]


def chunk_ids_for_doc(doc_id: str) -> List[str]:
    """All chunk ids whose source metadata came from this doc_id (a document
    can span several chunks after splitting)."""
    return [
        chunk_id
        for chunk_id, doc in zip(chunk_ids, docs_processed)
        if doc.metadata.get("doc_id") == doc_id
    ]


EVAL_SET = [
    {
        "query": query,
        "relevant_grades": {
            cid: 2
            for doc_id in doc_ids
            for cid in chunk_ids_for_doc(doc_id)
        },
    }
    for query, doc_ids in TEST_QUERIES
]
EVAL_SET = [example for example in EVAL_SET if example["relevant_grades"]]

missing = len(TEST_QUERIES) - len(EVAL_SET)
if missing:
    print(f"Warning: {missing} test doc_id(s) not found in docs_processed — did ingestion run?")

K = 5

RETRIEVAL_METHODS = {
    "dense_only": lambda q: [r["id"] for r in dense_search(q, K)],
    "sparse_only": lambda q: [r["id"] for r in sparse_search(q, K)],
    "hybrid_fused": lambda q: [r["id"] for r in hybrid_search_fused(q, top_k=K)],
    "hybrid_reranked": lambda q: [r["id"] for r in hybrid_search(q, top_k=K)],
}


def evaluate_method(method_fn, eval_set, k) -> float:
    scores = []
    for example in eval_set:
        ranked_ids = method_fn(example["query"])
        relevances = [example["relevant_grades"].get(doc_id, 0) for doc_id in ranked_ids]
        scores.append(ndcg_at_k(relevances, k))
    return sum(scores) / len(scores) if scores else 0.0


results_df = pd.DataFrame(
    [
        {"method": name, f"NDCG@{K}": evaluate_method(fn, EVAL_SET, K)}
        for name, fn in RETRIEVAL_METHODS.items()
    ]
)
results_df

,method,NDCG@5
0,dense_only,0.989965
1,sparse_only,0.938518
2,hybrid_fused,0.949234
3,hybrid_reranked,0.929756


### NDCG regression test
Runs after the whole notebook (ingestion + all retrieval strategies) and fails loudly if hybrid search's NDCG@5 drops below a floor — a real pass/fail check, not just a printed table. Re-run this cell any time after re-ingesting data or tweaking `RECIPROCAL_RANK_FUSION`/reranker settings to catch regressions.

In [16]:
NDCG_FLOOR = 0.7  # hybrid_reranked should reliably surface the right doc for these clear-cut queries

assert EVAL_SET, "EVAL_SET is empty — run the ingestion cells first so TEST_QUERIES' doc_ids exist."

failures = []
for _, row in results_df.iterrows():
    score = row[f"NDCG@{K}"]
    status = "PASS" if score >= NDCG_FLOOR else "FAIL"
    if row["method"] == "hybrid_reranked" and status == "FAIL":
        failures.append(f"{row['method']}: NDCG@{K}={score:.3f} < floor {NDCG_FLOOR}")
    print(f"[{status}] {row['method']:<16} NDCG@{K} = {score:.3f}")

assert not failures, "NDCG regression detected:\n" + "\n".join(failures)
print("\nAll NDCG checks passed.")

[PASS] dense_only       NDCG@5 = 0.990
[PASS] sparse_only      NDCG@5 = 0.939
[PASS] hybrid_fused     NDCG@5 = 0.949
[PASS] hybrid_reranked  NDCG@5 = 0.930

All NDCG checks passed.
